[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/12_compaction.ipynb)

# 12. Compaction

**Part 3: Context** · about 25 minutes

Capping each tool result (notebook 11) keeps every *piece* of the history small. But a long task makes many steps, and the history grows with each one. Sooner or later it no longer fits in the context window, or it fits but costs a fortune and the model starts losing track of what matters.

**Compaction** means shrinking the history in the middle of a run so the agent can keep going. There are two main ways to do it, and they fail in different ways:

| | Clearing | Summarising |
|---|---|---|
| What it does | Replaces old tool results with a short placeholder | Asks the model to write a summary of the work so far, and replaces old messages with it |
| Cost | Free | One extra model call |
| What gets lost | Every fact in the cleared results | Whatever the summary leaves out |

This notebook builds both, breaks the first on purpose, and shows the one rule that keeps either of them from corrupting the history.

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"


def chat(messages, **options):
    payload = {"model": llm.model, "messages": messages, **options}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def estimate_tokens(text):
    return max(1, len(text) // 4)


def history_tokens(messages):
    return sum(estimate_tokens((m.get("content") or "") + json.dumps(m.get("tool_calls") or [])) for m in messages)

## 1. The research agent, with a hook for compaction

The same task and notes as notebook 11, including the long transcript. The loop has one new parameter: `compact`, a function that gets the chance to shrink the history before every model call.

In [3]:
speakers = ["Ravi", "Meera", "Arjun", "Sana", "Kabir"]
topics = ["the vendor call", "the dashboard colours", "Friday's demo", "the office wifi", "the hiring plan",
          "the API docs", "the test flakiness", "the cafeteria menu"]
transcript = "\n".join(
    f"[10:{i % 60:02d}] {speakers[i % 5]}: Quick update on {topics[i % 8]}, nothing blocking, will follow up in chat."
    for i in range(300)
)

NOTES = {
    "standup.md": "Standup notes. The launch deadline moved to March 14 because of the audit.",
    "owners.md": "Launch owner: Priya. Backend owner: Sam.",
    "lunch.md": "The team voted for dosa on Friday.",
    "meeting_transcript.md": transcript,
}

TOOLS = [
    {"type": "function", "function": {"name": "list_files", "description": "List the names of all note files.",
                                      "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "read_file", "description": "Read one note file by its exact name.",
                                      "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                                                     "required": ["path"]}}},
]

QUESTION = "What is the launch deadline, and who owns the launch?"


def run_research(compact=None, max_steps=10):
    messages = [
        {"role": "system", "content": "Answer using the tools. Read the files you need, then reply briefly."},
        {"role": "user", "content": QUESTION},
    ]
    for step in range(1, max_steps + 1):
        if compact:
            messages = compact(messages)
        message = chat(messages, tools=TOOLS)["choices"][0]["message"]
        messages.append(message)
        if not message.get("tool_calls"):
            return {"answer": message.get("content"), "messages": messages, "steps": step}
        for call in message["tool_calls"]:
            path = json.loads(call["function"]["arguments"] or "{}").get("path")
            result = json.dumps(sorted(NOTES)) if call["function"]["name"] == "list_files" \
                else NOTES.get(path, f"Error: no file named {path!r}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return {"answer": None, "messages": messages, "steps": max_steps}


baseline = run_research()
print("Answer:", baseline["answer"])
print(f"History at the end: {len(baseline['messages'])} messages, about {history_tokens(baseline['messages']):,} tokens")

Answer: The launch deadline is March 14, and the owner is Priya.
History at the end: 11 messages, about 7,024 tokens


## 2. Clearing old tool results

The cheapest kind of compaction: keep the conversation's *shape*, meaning every message and every tool call, but replace the content of older tool results with a one-line placeholder. The model can still see *that* it read a file, just not what was in it.

Keeping the shape is important. A `tool` message must always follow the assistant message that called it, so deleting messages is risky. Replacing their content is always safe.

In [4]:
def clear_old_results(messages, keep_last=2):
    """Replace the content of all but the newest tool results with a short placeholder."""
    tool_indexes = [i for i, m in enumerate(messages) if m["role"] == "tool"]
    calls = {c["id"]: c for m in messages for c in (m.get("tool_calls") or [])}
    cleared = []
    for i, message in enumerate(messages):
        if i in tool_indexes[:-keep_last] if keep_last else i in tool_indexes:
            call = calls.get(message["tool_call_id"], {}).get("function", {})
            placeholder = (f"[cleared to save space: result of {call.get('name')}({call.get('arguments')}), "
                           f"about {estimate_tokens(message['content']):,} tokens]")
            message = {**message, "content": placeholder}
        cleared.append(message)
    return cleared


before = history_tokens(baseline["messages"])
after = history_tokens(clear_old_results(baseline["messages"], keep_last=1))
print(f"Clearing all but the last result: {before:,} -> {after:,} tokens")
print()
for message in clear_old_results(baseline["messages"], keep_last=1):
    if message["role"] == "tool":
        print(message["content"][:110])

Clearing all but the last result: 7,024 -> 287 tokens

[cleared to save space: result of list_files({}), about 16 tokens]
[cleared to save space: result of read_file({"path": "lunch.md"}), about 8 tokens]
[cleared to save space: result of read_file({"path": "meeting_transcript.md"}), about 6,785 tokens]
[cleared to save space: result of read_file({"path": "owners.md"}), about 10 tokens]
Standup notes. The launch deadline moved to March 14 because of the audit.


A huge saving for no model call at all. Now the catch: let's clear aggressively *while the agent is still working* and see whether it can still answer.

In [5]:
aggressive = run_research(compact=lambda messages: clear_old_results(messages, keep_last=1))
print("Answer:", aggressive["answer"])

Answer: The launch deadline is unknown, and the owner is unknown.


In [6]:
# Checkpoint: the offline script loses a fact it read but never wrote down.
if llm.offline:
    assert "unknown" in aggressive["answer"]
print("Checkpoint passed.")

Checkpoint passed.


The agent read the two files that contain the answer, and their results were cleared before it wrote the answer down. Both facts are just gone. (The offline script shows this every time. A real model might notice the placeholder and read the file again, which costs a step, or it might guess, which is worse.)

That's the trade-off with clearing. It's perfect for results whose job is done, like a test run that's been superseded by a newer one, or a file the agent has already edited. It destroys facts that only exist in the result.

## 3. Summarising

The other approach: before throwing old messages away, ask the model to write down what matters. The summary replaces the old messages, and the agent continues from it.

Two details decide whether this works.

**Where to cut.** We keep the system prompt, the original task, and the most recent messages. The kept tail must not start with a `tool` message, because a tool result whose call was summarised away is an orphan and most APIs reject it. So the cut point moves back until it lands somewhere safe.

**What to ask for.** The summary prompt says exactly what to keep: facts found, decisions, files changed and what's left to do. A vague "summarise this" produces a vague summary.

In [7]:
SUMMARY_PROMPT = """Summarize the work so far for the agent that will continue this task. Keep:
- every fact found that could matter for the task, with exact names, numbers and dates
- which files or tools were already used, so they aren't repeated
- what is left to do
Write short bullet points. Leave out anything that doesn't help finish the task."""


def safe_cut(messages, keep_recent):
    """Index where the kept tail starts, moved back so it never begins with a tool result."""
    cut = max(2, len(messages) - keep_recent)
    while 2 < cut < len(messages) and messages[cut]["role"] == "tool":
        cut -= 1
    return cut


def render(messages):
    """Turn messages into plain text for the summariser."""
    lines = []
    for message in messages:
        if message["role"] == "tool":
            lines.append(f"[tool result] {message['content'][:2000]}")
        elif message.get("tool_calls"):
            for call in message["tool_calls"]:
                lines.append(f"[assistant called {call['function']['name']}({call['function']['arguments']})]")
        else:
            lines.append(f"[{message['role']}] {message.get('content') or ''}")
    return "\n".join(lines)


def summarize(messages, keep_recent=2):
    head, cut = messages[:2], safe_cut(messages, keep_recent)
    middle, tail = messages[2:cut], messages[cut:]
    if not middle:
        return messages
    reply = chat([
        {"role": "system", "content": "You compress an AI agent's working history without losing facts."},
        {"role": "user", "content": f"{SUMMARY_PROMPT}\n\nTask: {messages[1]['content']}\n\n{render(middle)}"},
    ])
    summary = reply["choices"][0]["message"]["content"]
    note = {"role": "user",
            "content": f"Summary of the work so far (older messages were compacted to save space):\n{summary}"}
    return head + [note] + tail

## 4. Compact only when you need to

Summarising costs a model call, so we don't do it every step. We do it when the history passes a **budget**. Here we summarise everything after the task (`keep_recent=0`), because the huge transcript arrived in the most recent step. Agents usually keep the last few turns as they are, and you'll see why in the exercises. Here the budget is deliberately small, 1,500 tokens, so you can see it happen. Real agents usually compact at something like 70 to 80 percent of the context window.

In [8]:
BUDGET = 1500
events = []


def compact_when_needed(messages):
    size = history_tokens(messages)
    if size <= BUDGET:
        return messages
    compacted = summarize(messages, keep_recent=0)
    events.append((size, history_tokens(compacted)))
    print(f"    compacted: about {size:,} tokens -> {history_tokens(compacted):,}")
    return compacted


summarized = run_research(compact=compact_when_needed)
print("\nAnswer:", summarized["answer"])

note = next(m for m in summarized["messages"] if (m.get("content") or "").startswith("Summary of the work so far"))
print("\nThe summary the agent continued from:\n")
print(note["content"])

    compacted: about 7,011 tokens -> 144



Answer: The launch deadline is March 14, and the owner is Priya.

The summary the agent continued from:

Summary of the work so far (older messages were compacted to save space):
- Task: What is the launch deadline, and who owns the launch?
- [assistant called list_files({})]
- [tool result] ["lunch.md", "meeting_transcript.md", "owners.md", "standup.md"]
- [assistant called read_file({"path": "owners.md"})]
- [tool result] Launch owner: Priya. Backend owner: Sam.
- [tool result] Standup notes. The launch deadline moved to March 14 because of the audit.


In [9]:
# Checkpoint: compaction happened, and the answer survived it.
assert events, "The budget should have forced at least one compaction."
assert all(after < before for before, after in events)
if llm.offline:
    assert "March 14" in summarized["answer"] and "Priya" in summarized["answer"]
print("Checkpoint passed.")

Checkpoint passed.


Read the summary. Everything the agent needed is there, and the 300-line transcript is gone. The model kept "launch owner: Priya" because the prompt told it to keep facts that could matter for the task.

With a real model, read the summary critically. Did it keep exact dates and names, or paraphrase them? Did it note which files were already read? A summary that says "found some info about the launch" is how agents lose track halfway through long tasks.

## 5. Choosing a strategy

Production agents usually combine both, in this order:

1. **Cap every tool result** (notebook 11). Free, and prevents most problems.
2. **Clear results whose job is done**, such as old test runs or files that have since been edited. Free.
3. **Summarise when the budget is reached.** One model call, keeps the essentials.
4. **Write important facts somewhere permanent**, so they survive any compaction. That's notebook 14.

One cost to know about: compaction changes the beginning of the history, so any **prompt cache** built on the old history stops matching (Part 4 explains caching). That's another reason to compact rarely and in big steps, not a little on every call.

## Recap

- Compaction lets long tasks continue when the history gets too big.
- Clearing is free and keeps the history's shape, but loses facts that were only in the results.
- Summarising costs a call. Its quality depends on a prompt that says exactly what to keep.
- Never leave a tool result without its call. Cut the history only at safe points.
- Compact when you reach a budget, not on every step.

## Exercises

1. Make `clear_old_results` smarter: clear a `run_tests` result only when a newer one exists, and a file read only after that file has been edited.
2. Run the summarising agent three times on a real model and compare the summaries. Are they consistent? What would you change in `SUMMARY_PROMPT`?
3. Set `keep_recent=4` in `compact_when_needed` and rerun. Why does the history barely shrink? What would you do about big results that are still in the kept tail?
4. Add a check after summarising: ask a second, cheap model call whether the summary contains the answer to "who owns the launch". When should an agent verify its own summaries like this?

**Next:** [13. Code Search](13_code_search.ipynb). Finding the right code without reading every file.